In [43]:
from dotenv import load_dotenv
load_dotenv()

True

In [44]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent

In [45]:
loader = PyPDFLoader("../data/GenAI_Interview_Notes.pdf")
docs = loader.load()

In [46]:
splitters = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
splitted_docs = splitters.split_documents(docs)

In [47]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding = embeddings,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1071.19it/s]


In [58]:
@tool
def retriever_tool(query:str):
    """You are a tool that retriev all data related to the user query
    from docs to answer the user questions """
    docs = vector_store.similarity_search(query = query , k = 4)
    context = ""
    print("tool called \n")
    for doc in docs:
        context+=doc.page_content + "\n\n"
    return context


In [49]:
system_prompt = """
    You are a helpful assistant that answers questions using retrived context.
    Always use the 'retriever_tool' tool for questions requiring external knowledge
"""

In [50]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b"
)

In [59]:
agent = create_agent(
    model = llm,
    tools = [retriever_tool] ,
    system_prompt = system_prompt,
)

In [60]:
query = input("User: ")
response = agent.invoke({"messages":[{"role":"user","content":query}]})
result = response["messages"][-1].content
print(result)

tool called 

**Short Answers**

| Topic | Short Answer |
|-------|--------------|
| **GEN AI** | Generative AI (GenAI) is a branch of artificial intelligence that creates brand‑new content—text, images, audio, video, or code—by learning the statistical patterns of large datasets. |
| **RAG** | Retrieval‑Augmented Generation (RAG) is a technique that combines a generative language model with a retrieval system (e.g., a vector database) so the model can fetch relevant external information before producing an answer. |
| **RAG Pipeline** | The RAG pipeline follows these steps: (1) **Query** → (2) **Retrieval** of relevant documents or embeddings → (3) **Fusion** of retrieved evidence with the prompt → (4) **Generation** of a final answer. |

---

### 1. What is GEN AI?

**Short Answer:**  
GenAI is AI that *creates* new content rather than just classifying or predicting.  

**Detailed Answer:**  
- **Learning Phase**: A large neural network (usually a transformer) is trained on massive a